In [ ]:
import os































































import sys































































try:































































    from google.colab import drive































































    if not os.path.exists('/content/drive'):































































        drive.mount('/content/drive')































































    REPO_ROOT = '/content/drive/MyDrive/Stocks'































































except ImportError:































































    REPO_ROOT = os.getcwd()































































sys.path.insert(0, REPO_ROOT)































































os.chdir(REPO_ROOT)































































import importlib































































import portfolio.ranking































































importlib.reload(portfolio.ranking)































































from portfolio.ranking import build_ranking, WEIGHTS































































from IPython.display import display, HTML































































# =========================================================================































































# FETCH & RANK































































# =========================================================================































































































































































































results = build_ranking()































































































































# =========================================================================































































# RENDER HTML TABLE































































# =========================================================================































































html = []































































# CSS































































html.append("""<style>































































.rank-table { border-collapse: collapse; width: 100%; font-family: Arial, sans-serif; font-size: 12px; }































































.rank-table th { background: #2C3E50; color: white; padding: 8px 10px; text-align: left; position: sticky; top: 0; }































































.rank-table td { padding: 6px 10px; border-bottom: 1px solid #e0e0e0; color: #1a1a1a; }































































.rank-table tr:hover { filter: brightness(0.95); }































































.score-high { background: #C8E6C9; font-weight: bold; }































































.score-mid { background: #FFF9C4; }































































.score-low { background: #FFCDD2; }































































.rank-header { font-size: 18px; font-weight: bold; color: white; background: #2C3E50; padding: 12px 16px; border-radius: 6px 6px 0 0; }































































.rank-sub { font-size: 12px; color: #ccc; background: #2C3E50; padding: 0 16px 10px; border-radius: 0 0 6px 6px; margin-bottom: 10px; }































































.rank-legend { font-size: 11px; color: #555; margin-top: 12px; line-height: 1.8; }































































.rank-badge { display: inline-block; padding: 2px 8px; border-radius: 10px; font-size: 11px; font-weight: bold; }































































.badge-hf { background: #E3F2FD; color: #1565C0; }































































.badge-cy { background: #FFF3E0; color: #E65100; }































































.badge-ca { background: #F3E5F5; color: #6A1B9A; }































































</style>""")































































html.append('<div class="rank-header">Stock Ranking Dashboard</div>')































































html.append(f'<div class="rank-sub">Generated: {__import__("datetime").datetime.now().strftime("%Y-%m-%d %H:%M")} | Weights: Upside {int(WEIGHTS["upside"]*100)}% / Growth {int(WEIGHTS["growth"]*100)}% / Conviction {int(WEIGHTS["conviction"]*100)}% / Entry {int(WEIGHTS["entry"]*100)}% / Momentum {int(WEIGHTS["momentum"]*100)}%</div>')































































html.append('<table class="rank-table">')































































html.append('<tr>')































































html.append('<th>#</th><th>Ticker</th><th>What</th><th>Basket</th><th>Strategy</th>')































































html.append('<th>Price</th><th>Target</th><th>Upside</th><th>Rev Growth</th><th>EPS</th>')































































html.append('<th>Analysts</th><th>Consensus</th><th>Mkt Cap</th>')































































html.append('<th>52w Range</th>')































































html.append('<th>Fragility</th><th>Risk</th><th>Score</th>')































































html.append('<th>Upside</th><th>Growth</th><th>Conv.</th><th>Entry</th><th>Mom.</th><th>Profit</th><th>Frag.</th><th>Down.</th>')































































html.append('</tr>')































































def basket_bg(basket):































    colors = {































        "Core ETF": "#E3F2FD",































        "Nuclear": "#FFF8DC",































        "Quantum": "#F3E6F5",































        "Cyber": "#FFEBEE",































        "Industrial": "#E8EAF6",































        "SpecGrowth": "#E0F7FA",































        "MedTech": "#E8F5E9",































        "Defense": "#FFF3E0",































    }































    return colors.get(basket, "#FFFFFF")































































for r in results:































































    # Score color































































    score = r["composite"]































































    if score >= 60:































































        sc_class = "score-high"































































    elif score >= 40:































































        sc_class = "score-mid"































































    else:































































        sc_class = "score-low"































































    # Strategy badge































































    strat = r["strategy"]































































    if strat == "hold_forever":































































        badge = '<span class="rank-badge badge-hf">HOLD FOREVER</span>'































































    elif strat == "cycle":































































        badge = '<span class="rank-badge badge-cy">CYCLE</span>'































































    else:































































        badge = '<span class="rank-badge badge-ca">CATALYST</span>'































































    # Upside color































































    up = r["upside_pct"]































































    if up >= 50:































































        up_color = "#1B5E20"































































    elif up >= 20:































































        up_color = "#E65100"































































    elif up >= 0:































































        up_color = "#1a1a1a"































































    else:































































        up_color = "#B71C1C"































































    # Revenue growth color































































    rg = r["rev_growth_pct"]
    rg_missing = rg is None
    if rg_missing:
        rg = 0































































    if rg >= 50:































































        rg_color = "#1B5E20"































































    elif rg >= 20:































































        rg_color = "#E65100"































































    elif rg > 0:































































        rg_color = "#1a1a1a"































































    else:































































        rg_color = "#B71C1C"































































    # Yahoo link































































    yf_url = f'https://finance.yahoo.com/quote/{r["ticker"]}/'































































    ticker_link = f'<a href="{yf_url}" target="_blank" style="color:#1565C0; font-weight:bold;">{r["ticker"]}</a>'































































    # Market cap formatting































































    mc = r["market_cap_b"]































































    mc_str = f"${mc:.0f}B" if mc >= 1 else f"${mc*1000:.0f}M"































































    # 52w range































































    range_str = f"${r['low_52w']:.0f}-${r['high_52w']:.0f}" if r["low_52w"] else "—"































































    # Breakdown scores































































    bd = r.get("breakdown", {})































































    # Error handling































































    if r.get("error"):































































        html.append(f'<tr><td>{r["rank"]}</td><td>{ticker_link}</td><td colspan="22" style="color:#B71C1C;">Failed: {r["error"][:50]}</td></tr>')































































        continue































































    bg = basket_bg(r["basket"])































    html.append(f'<tr style="background:{bg};">')































































    html.append(f'<td><b>{r["rank"]}</b></td>')































































    html.append(f'<td>{ticker_link}</td>')































































    html.append(f'<td style="max-width:200px; font-size:11px;">{r["what"]}</td>')































































    html.append(f'<td>{r["basket"]}</td>')































































    html.append(f'<td>{badge}</td>')































































    html.append(f'<td>${r["price"]:,.2f}</td>')































































    html.append(f'<td>${r["target"]:,.2f}</td>' if r["target"] else '<td>—</td>')































































    html.append(f'<td style="color:{up_color}; font-weight:bold;">{up:+.0f}%</td>')































































    html.append(f'<td style="color:#999;">—</td>' if rg_missing else f'<td style="color:{rg_color};">{rg:+.0f}%</td>')































































    eps = r.get("eps")































































    eps_str = f"${eps:.2f}" if eps else "—"































































    eps_color = "#1B5E20" if eps and eps > 0 else "#B71C1C" if eps and eps < 0 else "#777"































































    html.append(f'<td style="color:{eps_color};">{eps_str}</td>')































































    html.append(f'<td style="text-align:center;">{r["num_analysts"]}</td>')































































    html.append(f'<td>{r["recommendation"]}</td>')































































    html.append(f'<td>{mc_str}</td>')































































    html.append(f'<td style="font-size:11px;">{range_str}</td>')































































    frag = r.get("fragility", "none")































































    frag_colors = {"none": "#4CAF50", "political": "#FF9800", "macro": "#FF9800", "binary": "#F44336"}































































    frag_labels = {"none": "—", "political": "⚠️ Political", "macro": "⚠️ Macro", "binary": "🔴 Binary"}































































    html.append(f'<td style="color:{frag_colors.get(frag, "#777")}; font-size:11px;">{frag_labels.get(frag, frag)}</td>')































































    ds = r.get("downside_if_fail", "low")































































    ds_colors = {"low": "#4CAF50", "moderate": "#FF9800", "severe": "#F44336", "zero": "#B71C1C"}































































    ds_labels = {"low": "-15%", "moderate": "-30-50%", "severe": "-70%+", "zero": "→ $0"}































































    html.append(f'<td style="color:{ds_colors.get(ds, "#777")}; font-weight:bold; font-size:11px;">{ds_labels.get(ds, ds)}</td>')































































    html.append(f'<td class="{sc_class}" style="text-align:center; font-size:14px;">{score:.0f}</td>')































































    html.append(f'<td style="text-align:center; font-size:10px;">{bd.get("upside", 0):.0f}</td>')































































    html.append(f'<td style="text-align:center; font-size:10px;">{bd.get("growth", 0):.0f}</td>')































































    html.append(f'<td style="text-align:center; font-size:10px;">{bd.get("conviction", 0):.0f}</td>')































































    html.append(f'<td style="text-align:center; font-size:10px;">{bd.get("entry", 0):.0f}</td>')































































    html.append(f'<td style="text-align:center; font-size:10px;">{bd.get("momentum", 0):.0f}</td>')































































    html.append(f'<td style="text-align:center; font-size:10px; color:{("#1B5E20" if bd.get("profitability", 0) > 0 else "#B71C1C" if bd.get("profitability", 0) < 0 else "#777")};">{bd.get("profitability", 0):+d}</td>')































































    html.append(f'<td style="text-align:center; font-size:10px; color:#B71C1C;">{bd.get("fragility", 0)}</td>')































































    html.append(f'<td style="text-align:center; font-size:10px; color:#B71C1C;">{bd.get("downside", 0)}</td>')































































    html.append('</tr>')































































html.append('</table>')































































# =========================================================================































































# LEGEND / FOOTNOTE































































# =========================================================================































































html.append('<div class="rank-legend">')































































html.append('<b>How this ranking works:</b>')































































html.append('<br>Each stock is scored on 5 positive factors (weighted 0-100), then adjusted with 3 risk modifiers (penalties/bonuses).')































































html.append('<br>')































































html.append('<br><b>POSITIVE FACTORS:</b>')































































html.append('<br><b>Upside (30%):</b> Distance from current price to analyst consensus target. Higher upside = higher score. 50%+ upside = max score.')































































html.append('<br><b>Revenue Quality (25%):</b> YoY revenue growth weighted by revenue scale. Growth from $1B base scores higher than same % from $1M base. Prevents tiny-base inflation.')































































html.append('<br><b>Conviction (15%):</b> Analyst consensus (Strong Buy=100, Buy=75, Hold=40, Sell=10) weighted by number of analysts. More analysts = more reliable.')































































html.append('<br><b>Entry Position (15%):</b> Where price sits in 52-week range. Near 52w low = 100 (good entry). Near 52w high = 0 (buying the top).')































































html.append('<br><b>Momentum (15%):</b> Trend alignment using 50-SMA and 200-SMA. Golden cross = 100. Death cross = 10.')































































html.append('<br>')































































html.append('<br><b>RISK ADJUSTMENTS:</b>')































































html.append('<br><b>Profitability:</b> Auto-fetched from Yahoo Finance. Profitable (EPS &gt; 0) = <span style="color:#1B5E20;">+5</span>. Unprofitable = <span style="color:#B71C1C;">-5</span>. Burning cash with &lt;2y runway = <span style="color:#B71C1C;">-8</span>.')































































html.append('<br><b>Thesis Fragility:</b> Manual tag. <span style="color:#4CAF50;">None</span> = 0 (monopoly/diversified). <span style="color:#FF9800;">Political/Macro</span> = -5 (depends on policy or commodity). <span style="color:#F44336;">Binary</span> = -10 (single pass/fail event).')































































html.append('<br><b>Downside Risk:</b> Manual tag. What happens if thesis fails? <span style="color:#4CAF50;">Low (-15%)</span> = 0. <span style="color:#FF9800;">Moderate (-30-50%)</span> = -5. <span style="color:#F44336;">Severe (-70%+)</span> = -8. <span style="color:#B71C1C;">Goes to $0</span> = -12.')































































html.append('<br>')































































html.append('<br><b>FORMULA:</b> Score = (Upside×30% + Growth×25% + Conviction×15% + Entry×15% + Momentum×15%) + Profitability + Fragility + Downside')































































html.append('<br>')































































html.append('<br><b>Score interpretation:</b> <span class="rank-badge" style="background:#C8E6C9;">60+</span> Strong candidate &nbsp; <span class="rank-badge" style="background:#FFF9C4;">40-59</span> Moderate &nbsp; <span class="rank-badge" style="background:#FFCDD2;">&lt;40</span> Weak / fully priced')































































html.append('<br>')































































html.append('<br><b>Strategy badges:</b> <span class="rank-badge badge-hf">HOLD FOREVER</span> = never sell, DCA monthly &nbsp; <span class="rank-badge badge-cy">CYCLE</span> = sell when growth decelerates (1-3y) &nbsp; <span class="rank-badge badge-ca">CATALYST</span> = sell on binary event (&lt;18m)')































































html.append('<br>')































































html.append('<br><b>To update:</b> Re-run this notebook — all data is fetched live. To add stocks, edit <code>RANKING_UNIVERSE</code> in <code>portfolio/ranking.py</code>. To change risk tags, edit <code>fragility</code> and <code>downside_if_fail</code> per stock.')































































html.append('<br>⚠️ This is NOT financial advice. Scores are mechanical — always do your own due diligence.')































































html.append('</div>')































































display(HTML('\n'.join(html)))































































































































